# Rüzgar Colab motoru

Bu defter **Rüzgar API**'yi Colab'da çalıştırır; PC'niz sadece arayüz olur.

1. **Runtime → Change runtime type → GPU** (önerilir)
2. Aşağıdaki hücreleri sırayla çalıştırın
3. Çıkan **ngrok https** adresini PC'de `scripts\Set-RuzgarColabUrl.ps1` ile kaydedin
4. PC'de `Ruzgar.ps1` — yerel CPU/Ollama yükü azalır

Ayrıntı: proje kökünde `COLAB_RUZGAR.md`

In [ ]:
# Proje yolu — Drive kullanıyorsanız PROJE_YOLU'nu düzenleyin
import os, shutil, subprocess, sys
from pathlib import Path

PROJE_YOLU = "/content/ruzgar"  # veya "/content/drive/MyDrive/CURSOR PROJELER/YAPAY ZEKA"
REPO_GIT = "https://github.com/umithaymana/ruzgar.git"

if not Path(PROJE_YOLU).joinpath("ilim-assistant", "desktop_server.py").is_file():
    if Path(PROJE_YOLU).is_dir():
        shutil.rmtree(PROJE_YOLU, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_GIT, PROJE_YOLU], check=True)
    print("GitHub'dan klonlandı:", PROJE_YOLU)
else:
    print("Proje zaten var:", PROJE_YOLU)

IA = Path(PROJE_YOLU) / "ilim-assistant"
assert IA.joinpath("desktop_server.py").is_file(), "ilim-assistant bulunamadı"
os.chdir(IA)
print("Çalışma dizini:", os.getcwd())

In [ ]:
# Bağımlılıklar (hafif — TTS yok)
!pip install -q -r requirements.txt uvicorn[standard] pyngrok httpx

In [ ]:
# API anahtarları + ngrok (Colab'da bir kez doldurun)
import os
from pathlib import Path

GEMINI_KEY = ""  # AI Studio: https://aistudio.google.com/apikey
GROQ_KEY = ""    # isteğe bağlı: https://console.groq.com
NGROK_TOKEN = "" # https://dashboard.ngrok.com/get-started/your-authtoken

# İsterseniz Drive'daki RUZGAR_BRAIN.env dosyasını okuyun:
brain = Path("RUZGAR_BRAIN.env")
if brain.is_file() and not GEMINI_KEY:
    for line in brain.read_text(encoding="utf-8").splitlines():
        if line.startswith("GLOBAL_API_KEY="):
            GEMINI_KEY = line.split("=", 1)[1].strip()
        if line.startswith("GROQ_API_KEY="):
            GROQ_KEY = line.split("=", 1)[1].strip()

assert GEMINI_KEY, "GEMINI_KEY veya RUZGAR_BRAIN.env gerekli"
assert NGROK_TOKEN, "NGROK_TOKEN gerekli"

os.environ["GLOBAL_API_KEY"] = GEMINI_KEY
os.environ["GOOGLE_GEMINI_API_KEY"] = GEMINI_KEY
if GROQ_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_KEY
os.environ["NGROK_AUTHTOKEN"] = NGROK_TOKEN
print("Anahtarlar yüklendi (Gemini:", bool(GEMINI_KEY), "Groq:", bool(GROQ_KEY), ")")

In [ ]:
# Rüzgar API + ngrok — bu hücre çalışır kalsın (durdurmayın)
import os
import subprocess
import sys

tok = os.environ.get("NGROK_AUTHTOKEN", NGROK_TOKEN)
subprocess.run(
    [sys.executable, "scripts/colab_start_ruzgar_api.py", "--ngrok-token", tok],
    check=False,
)